# Kubernetes Concepts for Scaling ML Deployments

Picture your model API packaged as a *container* — a self-contained, runnable box holding the app and everything it needs (the bridge below defines this precisely; Unit 4 teaches you to build one yourself). Traffic spikes — your one container can't keep up. Kubernetes (K8s) orchestrates multiple replicas of your container, routes traffic between them, and automatically scales up or down based on load.

**Note:** `kubectl` runs against a live cluster and can't execute inside a Jupyter kernel. This notebook teaches K8s concepts through YAML examples and explains each command's effect. Run the commands in a terminal connected to a K8s cluster (Minikube, Kind, or cloud).

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 5, lesson 01 "01. Big Data Concepts & Distributed Computing Theory" — scale-out, work distribution and fault tolerance across machines; Kubernetes applies those same ideas to replicas of your model API instead of partitions of a dataset.


## 📰 The day autoscaling could not save anyone: AWS us-east-1, October 2025

Late on **19 October 2025**, a latent race condition in DynamoDB's automated DNS management left the service's regional endpoint pointing at an **empty DNS record** in AWS's `us-east-1` region. Two automation processes ran concurrently, a stale plan overwrote a newer one, and cleanup automation then deleted it. DynamoDB's DNS was restored in about three hours — but the damage had propagated: EC2's instance-launch subsystem entered what AWS described as congestive collapse, and **new instances could not be launched** while it recovered. Full recovery took until the afternoon of **20 October — roughly 15 hours** in total, across dozens of services.

Read that through the lens of this notebook. A HorizontalPodAutoscaler responds to load by asking for more Pods. More Pods need nodes. Nodes need instances. **When the cloud cannot launch instances, "scale up" is a request that nobody answers** — and your carefully tuned `maxReplicas: 10` becomes a number in a YAML file.

**What goes wrong without this lesson.** One container on one machine has no answer to any of the questions production asks: who restarts it at 3 a.m., who adds capacity when traffic triples, who keeps traffic away from a Pod that is still loading a 500 MB model, and how a new version reaches users without a gap. Kubernetes answers all four declaratively — and it does so within limits that the outage above draws very clearly.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain what a Pod, Deployment, Service, and HPA are in plain terms
2. Read and write a Deployment YAML for an ML model container
3. Understand how a LoadBalancer Service routes traffic to Pods
4. Configure an HPA that scales 2-10 replicas based on CPU utilization
5. Explain zero-downtime rolling updates

## Containers in 60 Seconds (bridge — Docker is taught fully in Unit 4)

Kubernetes orchestrates *containers*, and this course teaches Docker itself in Unit 4 — two notebooks from now in the numbered path. You only need three definitions to follow everything below:

- **Image** — a frozen snapshot of an application plus everything it needs to run: OS libraries, Python, packages, your code, your model file. Named with a tag like `iris-classifier:v2` (`name:version`).
- **Container** — a *running instance* of an image. One image can be started as many identical containers — which is exactly what "scaling to 5 replicas" means.
- **Registry** — the store images are pushed to and pulled from (like PyPI, but for images). A cluster node pulls the image by name before starting a container from it.

That's the whole substrate: Kubernetes takes an image *name*, pulls it from a registry, and keeps N containers of it running. When a manifest below says `image: iris-classifier:v2`, read it as "run this frozen app snapshot". How to *build* such an image (Dockerfile, `docker build`, `docker run`) is Unit 4, notebook 01.

## 1. The Problem Kubernetes Solves

Plain Docker: you run one container. If it crashes, it's gone. If traffic doubles, you manually start another. You have to load-balance them yourself.

Kubernetes:
- Keeps your desired number of replicas running at all times (if one crashes, K8s starts a replacement)
- Routes incoming traffic across all healthy replicas
- Watches CPU/memory usage and adds or removes replicas automatically
- Deploys new versions without any downtime

## 2. Key Concepts

**Pod** — the smallest K8s unit. Usually wraps one container. If a Pod crashes, K8s schedules a new one. Pods are ephemeral: don't store state in them.

**Deployment** — declares *how many* Pods to run and *which container image* to use. The Deployment controller continuously reconciles reality with the declaration.

**Service** — a stable network address (IP + DNS name) that routes traffic to a set of Pods. Pods come and go; the Service IP stays fixed.

**HorizontalPodAutoscaler (HPA)** — watches CPU (or custom metrics) and adjusts the Deployment's replica count automatically.

In [1]:
# WHAT: create a working folder for the Kubernetes manifests we are about to write.
# WHY: K8s is configured declaratively through YAML files — this lesson builds
# them one by one so each concept (Deployment, Service, HPA) stays separate.
import os

YAML_DIR = "/tmp/k8s_manifests"
os.makedirs(YAML_DIR, exist_ok=True)
print(f"Writing K8s manifests to {YAML_DIR}")

Writing K8s manifests to /tmp/k8s_manifests


## 3. The Deployment YAML

A Deployment YAML is a declarative description of what you want to run. K8s continuously works to make the cluster match this declaration.

In [2]:
# WHAT: write the Deployment manifest — 3 replicas of the model-serving container.
# WHY: the Deployment is where availability policy lives: replica count, rolling-
# update rules, resource requests/limits, and the readiness/liveness probes.
# The manifest is stored as a Python string; the inline # notes inside the
# YAML explain what each field controls — read them top to bottom.
deployment_yaml = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
  labels:
    app: iris-classifier
spec:
  replicas: 3                          # Run 3 Pods simultaneously
  selector:
    matchLabels:
      app: iris-classifier             # This Deployment manages Pods with this label
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1                      # Allow 1 extra Pod during an update
      maxUnavailable: 0                # Never take a Pod down before a new one is ready
  template:
    metadata:
      labels:
        app: iris-classifier
    spec:
      containers:
        - name: iris-classifier
          image: myregistry/iris-classifier:v2   # Docker image to run
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"              # 0.25 CPU cores requested (used for scheduling)
              memory: "256Mi"
            limits:
              cpu: "500m"              # Hard cap: container cannot use more than 0.5 cores
              memory: "512Mi"
          readinessProbe:              # K8s won't send traffic until this passes
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 5
          livenessProbe:               # K8s restarts the container if this fails
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 10
"""

# Save the manifest; in a real cluster you would `kubectl apply -f` this file.
with open(f"{YAML_DIR}/deployment.yaml", "w") as f:
    f.write(deployment_yaml)

print(deployment_yaml)

apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
  labels:
    app: iris-classifier
spec:
  replicas: 3                          # Run 3 Pods simultaneously
  selector:
    matchLabels:
      app: iris-classifier             # This Deployment manages Pods with this label
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1                      # Allow 1 extra Pod during an update
      maxUnavailable: 0                # Never take a Pod down before a new one is ready
  template:
    metadata:
      labels:
        app: iris-classifier
    spec:
      containers:
        - name: iris-classifier
          image: myregistry/iris-classifier:v2   # Docker image to run
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"              # 0.25 CPU cores requested (used for scheduling)
              memory: "256Mi"
            limits:
              cpu: "500m"              # Hard cap: con

**Key fields to understand:**
- `replicas: 3` — K8s keeps exactly 3 Pods running. If one crashes, it starts a replacement.
- `resources.requests` — used by the scheduler to decide which node can fit this Pod.
- `resources.limits` — hard cap; container is killed if it exceeds this.
- `readinessProbe` — traffic only reaches a Pod once `/health` returns 200. This prevents requests going to a Pod that's still loading its model.
- `livenessProbe` — if `/health` fails repeatedly, K8s restarts the container.

## 4. The Service YAML

A Service gives your Pods a stable address. `LoadBalancer` type provisions a cloud load balancer (in AWS/GCP/Azure) that distributes traffic across all 3 Pods.

In [3]:
# WHAT: write the Service manifest — one stable address in front of all Pods.
# WHY: Pods come and go during scaling and updates; the Service gives clients a
# fixed endpoint and spreads their requests across whatever Pods exist right now.
service_yaml = """\
apiVersion: v1
kind: Service
metadata:
  name: iris-classifier-svc
spec:
  type: LoadBalancer                   # Provisions a cloud load balancer with an external IP
  selector:
    app: iris-classifier               # Routes to any Pod with this label
  ports:
    - name: http
      port: 80                         # External port (clients call :80)
      targetPort: 8000                 # Internal container port (your FastAPI app)
      protocol: TCP
"""

with open(f"{YAML_DIR}/service.yaml", "w") as f:
    f.write(service_yaml)

print(service_yaml)
print("After applying this Service, clients call: http://<EXTERNAL-IP>/predict")
print("K8s load-balances across all 3 Pod replicas automatically.")

apiVersion: v1
kind: Service
metadata:
  name: iris-classifier-svc
spec:
  type: LoadBalancer                   # Provisions a cloud load balancer with an external IP
  selector:
    app: iris-classifier               # Routes to any Pod with this label
  ports:
    - name: http
      port: 80                         # External port (clients call :80)
      targetPort: 8000                 # Internal container port (your FastAPI app)
      protocol: TCP

After applying this Service, clients call: http://<EXTERNAL-IP>/predict
K8s load-balances across all 3 Pod replicas automatically.


## 5. The HPA — HorizontalPodAutoscaler

In [4]:
# WHAT: write the HorizontalPodAutoscaler — scale between 2 and 10 Pods on CPU.
# WHY: traffic is never constant; the HPA turns scaling from a 3 a.m. manual
# task into a control loop with an explicit floor (availability) and cap (cost).
hpa_yaml = """\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-classifier              # The Deployment this HPA controls
  minReplicas: 2                       # Never scale below 2 (minimum availability)
  maxReplicas: 10                      # Never scale above 10 (cost cap)
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70       # Scale up when average CPU across Pods > 70%
                                       # Scale down when average CPU drops below 70%
"""

with open(f"{YAML_DIR}/hpa.yaml", "w") as f:
    f.write(hpa_yaml)

print(hpa_yaml)
print("HPA behavior:")
print("  CPU > 70%  → add Pods (up to maxReplicas=10)")
print("  CPU < 70%  → remove Pods (down to minReplicas=2)")
print("  Scale-down has a 5-minute cooldown to avoid flapping")

apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-classifier              # The Deployment this HPA controls
  minReplicas: 2                       # Never scale below 2 (minimum availability)
  maxReplicas: 10                      # Never scale above 10 (cost cap)
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70       # Scale up when average CPU across Pods > 70%
                                       # Scale down when average CPU drops below 70%

HPA behavior:
  CPU > 70%  → add Pods (up to maxReplicas=10)
  CPU < 70%  → remove Pods (down to minReplicas=2)
  Scale-down has a 5-minute cooldown to avoid flapping


## 6. Key kubectl Commands

Run these in a terminal connected to your cluster (after `kubectl config use-context <cluster>`).

In [5]:
# WHAT: the day-2 operations cheat sheet — the kubectl commands you actually run,
# with example output for each.
# WHY: deploy, inspect, watch autoscaling, roll out, roll back — reading the
# expected output now makes the real terminal much less mysterious later.
kubectl_commands = """
# --- Deploy everything ---
kubectl apply -f deployment.yaml
kubectl apply -f service.yaml
kubectl apply -f hpa.yaml

# --- Check Pod status ---
kubectl get pods
# NAME                              READY   STATUS    RESTARTS   AGE
# iris-classifier-7d9f6b8c4-4xqzp   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-k9v2d   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-rj3tf   1/1     Running   0          2m

# --- See the external IP for the Service ---
kubectl get service iris-classifier-svc
# NAME                    TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)        AGE
# iris-classifier-svc     LoadBalancer   10.96.142.55   34.117.12.200   80:32100/TCP   3m

# --- Tail logs from one Pod ---
kubectl logs -f iris-classifier-7d9f6b8c4-4xqzp

# --- Watch the HPA scaling decisions ---
kubectl get hpa iris-classifier-hpa --watch
# NAME                   REFERENCE                      TARGETS   MINPODS   MAXPODS   REPLICAS
# iris-classifier-hpa    Deployment/iris-classifier     42%/70%   2         10        3

# --- Manual scale override ---
kubectl scale deployment iris-classifier --replicas=5

# --- Check rollout status after updating the image ---
kubectl rollout status deployment/iris-classifier
# Waiting for rollout to finish: 1 out of 3 new replicas have been updated...
# Waiting for rollout to finish: 2 out of 3 new replicas have been updated...
# deployment "iris-classifier" successfully rolled out

# --- Roll back if the new version is broken ---
kubectl rollout undo deployment/iris-classifier
"""

print(kubectl_commands)


# --- Deploy everything ---
kubectl apply -f deployment.yaml
kubectl apply -f service.yaml
kubectl apply -f hpa.yaml

# --- Check Pod status ---
kubectl get pods
# NAME                              READY   STATUS    RESTARTS   AGE
# iris-classifier-7d9f6b8c4-4xqzp   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-k9v2d   1/1     Running   0          2m
# iris-classifier-7d9f6b8c4-rj3tf   1/1     Running   0          2m

# --- See the external IP for the Service ---
kubectl get service iris-classifier-svc
# NAME                    TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)        AGE
# iris-classifier-svc     LoadBalancer   10.96.142.55   34.117.12.200   80:32100/TCP   3m

# --- Tail logs from one Pod ---
kubectl logs -f iris-classifier-7d9f6b8c4-4xqzp

# --- Watch the HPA scaling decisions ---
kubectl get hpa iris-classifier-hpa --watch
# NAME                   REFERENCE                      TARGETS   MINPODS   MAXPODS   REPLICAS
# iris-classifier-hpa    Deploym

## 7. Rolling Update — Zero-Downtime Deployments

When you update the image tag in the Deployment YAML (e.g., `iris-classifier:v2` → `iris-classifier:v3`), K8s performs a rolling update:
1. Starts 1 new Pod with the new image (`maxSurge: 1` → temporarily 4 Pods exist)
2. Waits until the new Pod passes its readinessProbe
3. Terminates 1 old Pod (back to 3 total, but now 1 old + 2 new)
4. Repeats until all 3 are running the new version

Because `maxUnavailable: 0`, traffic is never sent to fewer than 3 ready Pods. Users never see downtime.

In [6]:
# WHAT: simulate the rolling-update timeline step by step as a printed table.
# WHY: maxSurge=1 with maxUnavailable=0 means capacity never dips below 3 ready
# Pods — this table is the proof that updates can drop zero requests.
# Visualize the rolling update timeline
import time

print("Rolling update timeline (replicas=3, maxSurge=1, maxUnavailable=0):")
print()
print(f"{'Step':<6} {'Old Pods':^12} {'New Pods':^12} {'Total':^8} {'Traffic goes to'}")
print("-" * 62)

# Each tuple is one moment: (step, old pods, new pods, where traffic goes).
steps = [
    ("0",  3, 0, "old pods only"),
    ("1",  3, 1, "old pods only (new pod not ready yet)"),
    ("2",  2, 1, "2 old + 1 new (new pod passed readiness)"),
    ("3",  2, 2, "2 old + 2 new"),
    ("4",  1, 2, "1 old + 2 new"),
    ("5",  1, 3, "1 old + 3 new"),
    ("6",  0, 3, "new pods only"),
]

for step, old, new, note in steps:
    total = old + new
    print(f"{step:<6} {old:^12} {new:^12} {total:^8} {note}")

print()
print("At every step, at least 3 ready Pods serve traffic.")
print("Zero requests are dropped during the entire update.")

Rolling update timeline (replicas=3, maxSurge=1, maxUnavailable=0):

Step     Old Pods     New Pods    Total   Traffic goes to
--------------------------------------------------------------
0           3            0          3     old pods only
1           3            1          4     old pods only (new pod not ready yet)
2           2            1          3     2 old + 1 new (new pod passed readiness)
3           2            2          4     2 old + 2 new
4           1            2          3     1 old + 2 new
5           1            3          4     1 old + 3 new
6           0            3          3     new pods only

At every step, at least 3 ready Pods serve traffic.
Zero requests are dropped during the entire update.


## 💬 Discuss

The rolling-update table above is the experiment: with `replicas: 3`, `maxSurge: 1`, `maxUnavailable: 0`, the printed timeline never drops below 3 ready Pods at any step. That is the guarantee. Argue about its price and its edges.

1. `maxUnavailable: 0` guarantees no capacity loss and makes every rollout slower and briefly more expensive (4 Pods instead of 3). For a model API serving internal dashboards at 09:00, would you keep it? What would you change for a service where a 30-second gap is genuinely acceptable?
2. Traffic reaches a Pod only after `/health` returns 200. Our `/health` returns `{"status": "ok"}` as soon as the process starts. If loading the model takes 40 seconds, what does the readiness probe actually certify — and what would a caller experience during a rollout? Write the probe you would use instead.
3. The HPA scales on CPU at 70%. For a model whose latency is dominated by waiting on a feature store, CPU stays low while requests queue. Name the metric you would scale on instead, and say what new failure mode you have just introduced by scaling on it.

## 8. Writing All Manifests to Disk

In [7]:
# Verify all three YAML files were written
import os
for fname in os.listdir(YAML_DIR):
    path = os.path.join(YAML_DIR, fname)
    size = os.path.getsize(path)
    print(f"{fname}: {size} bytes")

print(f"\nApply all with: kubectl apply -f {YAML_DIR}/")
print("K8s will create the Deployment, Service, and HPA in that order.")

deployment.yaml: 1511 bytes
service.yaml: 458 bytes
hpa.yaml: 675 bytes

Apply all with: kubectl apply -f /tmp/k8s_manifests/
K8s will create the Deployment, Service, and HPA in that order.


## 9. K8s vs Plain Docker

| Capability | Plain Docker | Kubernetes |
|---|---|---|
| Run 1 container | Yes | Yes |
| Auto-restart on crash | No (need docker restart policy) | Yes (always) |
| Load balance across replicas | No (need nginx/haproxy) | Yes (Service) |
| Auto-scale on CPU | No | Yes (HPA) |
| Zero-downtime update | No | Yes (rolling update) |
| Multi-node scheduling | No | Yes |
| Health-gate traffic | No | Yes (readinessProbe) |

## 10. Summary

In this notebook you:
- Learned the four core K8s objects: Pod, Deployment, Service, HPA
- Wrote a Deployment YAML with resource limits, readiness probes, and a rolling update strategy
- Wrote a LoadBalancer Service YAML that routes port 80 to container port 8000
- Wrote an HPA YAML that scales 2-10 replicas at 70% CPU target
- Traced through a rolling update showing zero Pods are unavailable at any step

**K8s in one sentence:** you declare what you want; K8s continuously makes the cluster match your declaration, handling failures and scaling automatically.

## Self-Check (answer before scrolling up)

1. **What is the difference between a Pod and a Deployment?** Why do you almost never create Pods directly?
2. **What does the HPA do when CPU drops below 70% for an extended period?** Is this instant? Why or why not?
3. **What is a rolling update and why is it better than stopping all replicas before starting new ones?** Which YAML field guarantees no Pod is taken down before a replacement is ready?

## ⚠️ Where this breaks

- **The HPA cannot conjure capacity.** It changes a replica count. If the cluster has no room, Pods sit `Pending`; if the cloud region cannot launch instances — as in `us-east-1` on 20 October 2025 — nothing helps. Autoscaling handles *demand* variation, not *supply* failure. Multi-region is the answer to supply failure, and it is a different, much more expensive design.
- **CPU is usually the wrong scaling signal for model serving.** It is the default because it is always available. If your service is I/O-bound, or if inference is fast but queueing is slow, CPU utilisation stays flat while p99 latency climbs. Scale on requests-per-Pod or on queue depth when you can measure them.
- **Scaling is not instant, and cold starts are the hidden cost.** Between "CPU crosses 70%" and "a new Pod serves traffic" sit: HPA's evaluation interval, scheduling, image pull, container start, model load, and readiness probe. For a large artifact that is minutes. A traffic spike that lasts 90 seconds will be over before the extra Pods arrive.
- **The assumption that must hold:** Pods are stateless and interchangeable. Anything a replica keeps in memory — a cache, a rate-limit counter, an in-process log buffer — is per-Pod and disappears on every scale-down and every rollout. The in-memory rate limiter from Unit 3 notebook 05 is a concrete example that breaks the moment you run more than one replica.
- **Readiness probes only certify what they check.** A probe that returns 200 from a process that has not finished loading its model will happily route real traffic into a `503`. Make the probe assert the thing you care about — that the model is loaded and can score the card's sample row.
- **The cheaper alternative.** Kubernetes has a real operational cost: manifests, an upgrade treadmill, and a class of failures your team must learn to debug. For a single model with predictable traffic, one container behind a managed load balancer — or a serverless container platform that scales to zero — is less machinery and fewer ways to be wrong. Adopt Kubernetes when you have several services to orchestrate, not because it is what production is supposed to look like.

## 📚 References

1. Verma, A., Pedrosa, L., Korupolu, M. R., Oppenheimer, D., Tune, E., & Wilkes, J. (2015). *Large-Scale Cluster Management at Google with Borg*. EuroSys. <https://research.google/pubs/large-scale-cluster-management-at-google-with-borg/>
2. Burns, B., Grant, B., Oppenheimer, D., Brewer, E., & Wilkes, J. (2016). *Borg, Omega, and Kubernetes*. ACM Queue 14(1). <https://research.google/pubs/borg-omega-and-kubernetes/>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access. <https://arxiv.org/abs/2205.02302>
